## Problem Statement
A healthcare organization wants to improve the early identification of patients at *risk of heart disease*. The organization wants to build an *intelligent classification system* that can predict the likelihood of heart disease based on a patient's medical and clinical characteristics.

In this project, **Logistic Regression** is used to learn the relationship between the available medical features and the `target` variable.

The model predicts one of two classes:

- `0` → Negative class
- `1` → Positive class

In [1]:
#importing libraries
import numpy as np
from collections import Counter
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import StandardScaler


print("Libraries imported successfully...Bliss")

Libraries imported successfully...Bliss


In [2]:
# Loading dataset
path = "/kaggle/input/datasets/dharanireddy/heart-disease/heart.csv"
df = pd.read_csv(path)
print("Dataset loaded successfully")

Dataset loaded successfully


## Explore the Dataset

#### Dataset Description

The dataset contains medical and clinical attributes that are used to predict the `target` variable using Logistic Regression.

| Column | Description | Type |
|---|---|---|
| `age` | Age of the patient in years | Numerical |
| `sex` | Sex of the patient | Categorical/Binary |
| `cp` | Type of chest pain experienced by the patient | Categorical |
| `trestbps` | Resting blood pressure of the patient | Numerical |
| `chol` | Serum cholesterol level | Numerical |
| `fbs` | Indicates whether fasting blood sugar is greater than 120 mg/dl | Binary |
| `restecg` | Resting electrocardiographic results | Categorical |
| `thalach` | Maximum heart rate achieved | Numerical |
| `exang` | Indicates whether exercise-induced angina is present | Binary |
| `oldpeak` | ST depression induced by exercise relative to rest | Numerical |
| `slope` | Slope of the peak exercise ST segment | Categorical |
| `ca` | Number of major vessels colored by fluoroscopy | Numerical |
| `thal` | Thalassemia-related measurement/category | Categorical |
| `target` | Target class indicating the predicted outcome | Binary |

### Target Variable

The `target` column is the dependent variable that the Logistic Regression model attempts to predict.

- `0` → Negative class
- `1` → Positive class

The remaining columns are used as input features for the model.

In [3]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [4]:
df.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


In [6]:
ls = df["target"].unique()
print("Unique values in target :", ls)

Unique values in target : [1 0]


In [7]:
print("Binary values columns:")
for i in df.columns:
    if df[i].nunique() < 3:
        print(i)

Binary values columns:
sex
fbs
exang
target


In [8]:
print(f"Rows : {df.shape[0]}")
print(f"columns : {df.shape[1]}")

Rows : 303
columns : 14


## Define Features and Target

The dataset is divided into two components:

### Independent Variables — `X`

`X` contains all input features used by the Logistic Regression model.

The `target` column is removed because it is the variable that the model needs to predict.

### Dependent Variable — `y`

`y` contains the `target` values that represent the classification outcome.

In [9]:
X = df.drop("target", axis=1)
y = df["target"]

In [10]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2


In [11]:
y.head()

0    1
1    1
2    1
3    1
4    1
Name: target, dtype: int64

In [12]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print("Dataset split successfully")

Dataset split successfully


In [13]:
X_train.head()
y_train.head()

132    1
202    0
196    0
75     1
176    0
Name: target, dtype: int64

In [14]:
X_test.head()
y_test.head()

179    0
228    0
111    1
246    0
60     1
Name: target, dtype: int64

In [15]:
# Check the ratio of targeted values

for i in ls:
    print(f"NO. of {i} : {np.int64(y_train[y_train == i].count())}")

NO. of 1 : 133
NO. of 0 : 109


## Train a Logistic Model

Logistic Regression is a supervised ML algorithm commonly used for binary classification.

Instead of directly predicting a continuous value, Logistic Regression estimates the probability of an observation belonging to a particular class.

The model is trained using the training features and corresponding target values.

The parameter: `max_iter = 10000`

In [16]:
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=10000)

## Generate Predictions

After training the Logistic Regression model, predictions are generated using the testing features.

The predicted values are stored in `y_pred`.

These predictions are then compared with the actual values in `y_test` to evaluate the model.

In [17]:
y_pred = model.predict(X_test)
print("Values predicted successfully...Bliss")

Values predicted successfully...Bliss


In [18]:
print("accuracy: ", accuracy_score(y_test, y_pred)*100, "%")
print("precision: ", precision_score(y_test, y_pred)*100, "%")

accuracy:  88.52459016393442 %
precision:  87.87878787878788 %


In [19]:
arr = np.array(y_pred)

values, counts = np.unique(arr, return_counts=True)


print(f"No. of {values[0]} : {counts[0]}")
print(f"No. of {values[1]} : {counts[1]}")

No. of 0 : 28
No. of 1 : 33


In [20]:
# Evluation Matrices
ls = [accuracy_score, precision_score, recall_score, f1_score]

print("Confusion Matrix :")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("-"*50)
for i in ls:
    score = i(y_test, y_pred)
    print(f"{i.__name__} : {score*100} %")

Confusion Matrix :
[[25  4]
 [ 3 29]]
--------------------------------------------------
accuracy_score : 88.52459016393442 %
precision_score : 87.87878787878788 %
recall_score : 90.625 %
f1_score : 89.23076923076924 %


## Feature Scaling

The dataset contains numerical features with different scales.

For example, some variables have values in relatively small ranges, while others have larger numerical values.

`StandardScaler` is applied to standardize the training and testing features.

The scaler is:

1. Fitted on the training data.
2. Used to transform the training data.
3. Used to transform the testing data.

This keeps the scaling process based on the training dataset.

In [21]:
scalar = StandardScaler()

X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## Logistic Regression After Feature Scaling

After applying standardization, the Logistic Regression model is trained again using the scaled training data.

Predictions are then generated using the scaled test data.

The resulting accuracy and precision are compared with the earlier model results.

In [22]:
print("accuracy :", accuracy_score(y_test, y_pred)*100, "%")
print("precision :", precision_score(y_test, y_pred)*100, "%")

accuracy : 85.24590163934425 %
precision : 87.09677419354838 %


## Conclusion

This project successfully developed a **Logistic Regression model** to classify patients based on their heart disease-related medical information.

### Model Performance

Before feature scaling, the model achieved:

| **Metric** | **Score** |
|---|---:|
| **Accuracy** | **88.52%** |
| **Precision** | **87.88%** |
| **Recall** | **90.63%** |
| **F1 Score** | **89.23%** |

### Feature Scaling

Feature scaling was applied to standardize the input features. However, on this dataset, the scaled model did not improve the reported test performance. Therefore, the **unscaled model was retained for the final results**.

### Key Outcome

The results demonstrate that the model can identify meaningful patterns in patient data and provide useful predictions for the target class. Such a system can help healthcare professionals **identify patients who may require further medical evaluation**, supporting earlier attention and data-driven decision-making.

> **Note:** This model is intended as a **supporting tool** and should not be considered a replacement for professional medical diagnosis.